# Turbidity and Chlorophyll-A time series monitoring

* **Products used:** 
[wq_annual](https://explorer.digitalearth.africa/products/wq_annual), 
[ls5_sr](https://explorer.digitalearth.africa/products/ls5_sr),
[ls7_sr](https://explorer.digitalearth.africa/products/ls7_sr),
[ls8_sr](https://explorer.digitalearth.africa/products/ls8_sr),
[ls9_sr](https://explorer.digitalearth.africa/products/ls9_sr),
[s2_l2a](https://explorer.digitalearth.africa/products/s2_l2a)

## Background
The Water Quality Monitoring Service (WQMS) dataset notebook introduces the several variables available in the annual water quality product. This notebook shows how to monitor the Total Suspended Matter (TSM), a proxy for the turbidity, and Chlorophyll-A, on an individual scene basis and aggregate into monthly spatial medians.

## Description
A _compulsory_ description of the notebook, including a brief overview of how Digital Earth Africa helps to address the problem set out above.
It can be good to include a run-down of the tools/methods that will be demonstrated in the notebook:

1. First we do this
2. Then we do this
3. Finally we do this

***

## Getting started

To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell. 

### Load packages
Import Python packages that are used for the analysis.

In [2]:
%matplotlib inline

import datacube
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from deafrica_tools.plotting import display_map
from deafrica_tools.datahandling import load_ard

import sys
sys.path.append("/home/jovyan/deafrica-sandbox-notebooks/deafrica_water_quality/src")
from water_quality.mapping.fai import FAI
from water_quality.mapping.algorithms import set_wq_algorithms, run_wq_algorithms, harmonize_wq_variables, normalize_wq_variables, compute_trophic_state_index
from water_quality.mapping.pixel_correction import R_correction

### Connect to the datacube

Connect to the datacube so we can access DE Africa data.
The `app` parameter is a unique name for the analysis which is based on the notebook file name.

In [3]:
dc = datacube.Datacube(app='TSM_Chla_monitoring')

### Analysis parameters

An *optional* section to inform the user of any parameters they'll need to configure to run the notebook:

* `param_name_1`: Simple description (e.g. `example_value`). Advice about appropriate values to choose for this parameter.
* `param_name_2`: Simple description (e.g. `example_value`). Advice about appropriate values to choose for this parameter.


In [3]:
param_name_1 = 'example_value'
param_name_2 = 'example_value'

## Load in annual data

> **Note:** Use this markdown format (sparingly) to draw particular attention to an important point or caveat

In [4]:
query = {
    "product": "wq_annual",
    "x": (35.72, 35.92),
    "y": (-3.90, -3.35), # Lake Manyara
    "crs": "EPSG:4326",
    "time": ("2020-01-01", "2025-12-31"),
    "output_crs": "EPSG:6933",
    "resolution": (-10, 10),
    "measurements": [
        "tsm",           # Turbidity
        "chla",          # Chlorophyll-A
        "tsi",           # Trophic State Index
        "clear_water",   # Clear water mask
        "water_mask",    # Water presence mask
    ]
}

In [5]:
display_map(x=query["x"], y=query["y"])

In [6]:
ds = dc.load(**query)

## Apply water quality algorithm to individual timesteps

### Load satellite scenes in time period

In [16]:
measurements = {
    'oli': {
        'blue': 'oli02',
        'green': 'oli03',
        'red': 'oli04',
        'nir': 'oli05',
        'swir_1': 'oli06',
        'swir_2': 'oli07',
        'pixel_quality': 'oli_pq'
    },
    'msi': {
        'blue': 'msi02',
        'green': 'msi03',
        'red': 'msi04',
        'red_edge_1': 'msi05',
        'red_edge_2': 'msi06',
        'red_edge_3': 'msi07',
        'nir': 'msi08',
        'nir_narrow': 'msi8a',
        'swir_1': 'msi11',
        'swir_2': 'msi12',
        'qa': 'msi_pq'
    },
    'tm': {
        'blue': 'tm01',
        'green': 'tm02',
        'red': 'tm03',
        'nir': 'tm04',
        'swir_1': 'tm05',
        'swir_2': 'tm07',
        'pixel_quality': 'tm_pq'
    }
}

In [9]:
# the place and times of interest can be chosen from here -- other places can be added 
def set_place_and_time(placename='Hartbeespoort_dam',year1='2024-01-01',year2='2025-10-31'):
    places = {
        'Hartbeespoort_dam':   {"xyt" :{"x": (  27.7972, 27.91117), "y" : (-25.7761,-25.7275) ,"time": (year1,year2)},"desc": "Haartbeesport Dam  -- South Africa"},
        'Lake_Manyara':        {"xyt" :{"x": ( 35.724 ,  35.929 ), "y" : ( -03.814, -03.409), "time": (year1,year2) },"desc": "Lake_Manyara, Tanzania"          },#this is the lake to use as an example of monitoring, see 2015-12-28
        'Lake_Baringo'     :   {"xyt" :{"x": (36.00,  36.17),     "y": (00.45,00.74),        "time": (year1,year2)},"desc":'Lake Baringo'    },
        'Weija_Reservoir'  :   {"xyt" :{"x": (-0.325, -0.41),     "y": ( 5.54, 5.62),        "time": (year1,year2)  },"desc":''                },
        'Lake_Sulunga'     :   {"xyt" :{"x": (34.95, 35.4),       "y": (-6.3, -5.8),         "time": (year1,year2)  },"desc":""                },
        'cameroon_res1'    :   {"xyt" :{"y": (6.20,6.30),          "x": (11.25, 11.35),      "time": (year1,year2)  },"desc": "reservoir in cameroon"},
        'Lake_vic_algae':      {"xyt" :{"x": ( 34.62, 34.78),       "y" : ( -.18,-.08),      "time": (year1,year2)  },"desc": "Lake Victoria Water Hyacinth affected area in NE, port Kisumu"},
        'Ethiopia_Lake_Tana':  {"xyt" :{"x": ( 37.05,   37.22),    "y" : (  11.9  ,  12.0),  "time": (year1,year2)  },"desc": "Ethiopia_Lake_Tana"          },
        'Mare_aux_Vacoas':     {"xyt" :{"x": ( 57.485,  57.524),   "y" : ( -20.389, -20.359),"time": (year1,year2)  },"desc": "Mare_aux_Vacoas"          },
        'SA_smalldam':         {"xyt" :{"x": ( 19.494,  19.498),   "y" : ( -33.802, -33.800),"time": (year1,year2)  },"desc": "Irrigation Dam, South Africa"          },
        'Lake Chamo'   :       {"xyt" :{"x": ( 37.45,   37.65) ,   "y" : (   5.685 ,  5.979), "time": (year1,year2)  },"desc": "Lake Chamo, Ethiopia"          },
        'Lake Ziway'   :       {"xyt" :{"x": ( 38.711,  38.966),   "y" : (   7.838 ,  8.148), "time": (year1,year2)  },"desc": "Lake Ziway, Ethiopia"          },
        'Lake Alwassa' :       {"xyt" :{"x": ( 38.380,  38.493),   "y" : (   6.977 ,  7.133), "time": (year1,year2)  },"desc": "Lake Alwassa, Ethiopia"          },
        'Lake Elmenteita' :    {"xyt" :{"x": ( 36.211,  36.273),   "y" : (  -0.488 , -0.401), "time": (year1,year2)  },"desc": "Lake Elmenteita, Kenya"          },
        'Farihy_itasy':        {"xyt" :{"x": ( 46.73 ,  46.83 ),   "y" : ( -19.10 , -19.04 ),"time": (year1,year2)},"desc": "Farihy Itasy, Madagascar"          },
        'Kolokonda':           {"xyt" :{"x": ( 35.4888, 35.5488),   "y" : ( -5.976, -5.916 ),"time": (year1,year2)},"desc": "Kolokonda, Tanzania"          },
        'Tana_Hayk'      :     {"xyt" :{"x": (  36.95 ,  37.65),   "y" : ( 11.56 , 12.33   ) ,"time": (year1,year2)},"desc": "T'ana Hayk', northern Ethiopia"},
        'Lake_Malawi'    :     {"xyt" :{"x": (  34.25 ,  34.97),   "y" : ( -13.6 , -13.3    ) ,"time": (year1,year2)},"desc": "Lake Malawi - part of"},
        'Lago de Cabora' :     {"xyt" :{"x": (  30.90 ,  32.52),   "y" : ( -15.95, -15.45  ) ,"time": (year1,year2)},"desc": "Lago de Cabora Basa - Mozambique"},
        'Mtera Reservoir':     {"xyt" :{"x": (  35.60 ,  36.01),   "y" : ( - 7.20, - 6.86  ) ,"time": (year1,year2)},"desc": "Lake Nzuhe, Tanzania"},
        'Barrage Joumine':     {"xyt" :{"x": (  09.53 ,  09.62),   "y" : ( 36.952,  37.00  ) ,"time": (year1,year2)},"desc": "Joumine Dam,Tunisia"},
        'Tunisia_Dam'    :     {"xyt" :{"x": (  08.53 ,  08.56),   "y" : ( 36.685,  36.75  ) ,"time": (year1,year2)},"desc": "Tunisia"},
        'Lake_Ngami'     :     {"xyt" :{"x": (  22.55 ,  22.89),   "y" : ( - 20.6, -20.37  ) ,"time": (year1,year2)},"desc": "Botswana"},
        'Lake_Chilwa'    :     {"xyt" :{"x": (  35.5 ,  35.9),   "y" : ( - 15.6, -14.90  ) ,"time": (year1,year2)},"desc": "Malawi - Lake Chilwa"},
        'Lake_Malombe'   :     {"xyt" :{"x": (  35.15 ,  35.35),   "y" : ( - 14.8, -14.50  ) ,"time": (year1,year2)},"desc": "Malawi - Lake Malombe"},
        'Lake_Piti'      :     {"xyt" :{"x": (  32.85 ,  32.90),   "y" : ( - 26.6, -26.50  ) ,"time": (year1,year2)},"desc": "Mozambique - Lake Piti"},
        'Maputo_reserve' :     {"xyt" :{"x": (  32.79 ,  32.83),   "y" : ( - 26.55, -26.50  ) ,"time": (year1,year2)},"desc": "Mozambique - Maputo reserve"},
        'Indian_Ocean'   :     {"xyt" :{"x": (  57.75 ,  57.80),   "y" : ( - 20.5 , -20.45  ) ,"time": (year1,year2)},"desc": "Mauritius - Oceanic waters"},
        'Mare_Vacoas'    :     {"xyt" :{"x": (  57.48 ,  57.52),   "y" : ( - 20.38 , -20.36  ) ,"time": (year1,year2)},"desc": "Mauritius - Mare aux Vacoas"},
        'Naute'          :     {"xyt" :{"x": (  17.93 ,  18.05),   "y" : ( - 26.97 , -26.92  ) ,"time": (year1,year2)},"desc": "Namibia - Naute reserve"},
        'Lake_Turkana'   :     {"xyt" :{"x": (  35.80 ,  36.72),   "y" : (    2.38 ,   4.79  ) ,"time": (year1,year2)},"desc": "Kenya -- Lake Turkana"},
        'Lake Bogoria'   :     {"xyt" :{"x": (  36.058, 36.133),   "y" : (  0.1791 ,0.3534) ,"time": (year1,year2)},"desc": "Lake Bogoria -- Tanzania"},
        }
    # --- send back the dictionary for the place of interest
    place_and_time = {}
    place_and_time['placename'] = placename
    place_and_time['xyt']       = places[placename]['xyt']
    place_and_time['desc']      = places[placename]['desc']
    return(place_and_time)

In [7]:
def set_analysis_parameters(place_and_time, max_cells=10000, verbose=True):
    """
    Determine spatial analysis parameters based on a given space-time extent.

    The function computes an appropriate grid resolution such that the total
    number of spatial cells does not exceed `max_cells`. It also selects an
    appropriate resampling method based on the resulting resolution.

    Parameters
    ----------
    place_and_time : dict
        Dictionary containing:
        - 'placename': Name of the location
        - 'desc': Description of the analysis
        - 'xyt': Dictionary with spatial and temporal bounds:
            - 'x': [xmin, xmax] (longitude)
            - 'y': [ymin, ymax] (latitude)
            - 'time': [start_time, end_time]

    max_cells : int, optional
        Maximum number of spatial grid cells allowed (default = 10,000)

    verbose : bool, optional
        If True, prints a summary of computed parameters

    Returns
    -------
    params : dict
        Dictionary containing computed analysis parameters
    """

    # ---------------------------------------------------------
    # 1. Define allowable grid cell size range (in metres)
    # ---------------------------------------------------------
    cell_min = 10    # minimum resolution (fine)
    cell_max = 500   # maximum resolution (coarse)

    # Extract spatial + temporal domain
    xyt = place_and_time['xyt']
    print(xyt)

    # ---------------------------------------------------------
    # 2. Extract spatial bounds
    # ---------------------------------------------------------
    x0, x1 = xyt['x']   # longitude range
    y0, y1 = xyt['y']   # latitude range

    # ---------------------------------------------------------
    # 3. Approximate spatial extent in metres
    #    - Longitude scaled by cos(latitude) due to Earth's curvature
    # ---------------------------------------------------------
    dxm = ((x1 - x0) * 100000) * np.cos(y0 * np.pi / 180.0)
    dym = ((y1 - y0) * 100000)

    # Total area (m²)
    dAm = abs(dxm * dym)

    # ---------------------------------------------------------
    # 4. Compute grid resolution based on max_cells constraint
    # ---------------------------------------------------------
    # Ideal cell size (square root distributes area evenly)
    cell_dxm = (dAm / max_cells) ** 0.5

    # Clamp resolution to allowable range
    cell_dxm = np.max([cell_dxm, cell_min])
    cell_dxm = np.min([cell_dxm, cell_max])

    # Round to nearest 10 metres for stability
    cell_dxm = int(cell_dxm / 10) * 10

    # Final grid resolution (square cells)
    grid_resolution = (cell_dxm, cell_dxm)

    # Estimated number of cells
    cellcount = dAm / (cell_dxm ** 2)

    # ---------------------------------------------------------
    # 5. Additional derived properties
    # ---------------------------------------------------------
    aspect_ratio = np.abs(dym / dxm)          # shape of region
    cell_area = (cell_dxm ** 2) / 1_000_000   # km² per cell

    # ---------------------------------------------------------
    # 6. Choose resampling strategy
    #    - Coarse grid → nearest (faster, less smoothing)
    #    - Fine grid   → bilinear (smoother)
    # ---------------------------------------------------------
    if cell_dxm > 60:
        resampling_option = "nearest"
    else:
        resampling_option = "bilinear"

    # ---------------------------------------------------------
    # 7. Extract temporal range (years only)
    # ---------------------------------------------------------
    y1, y2 = pd.DatetimeIndex([
        xyt['time'][0],
        xyt['time'][1]
    ]).year[[0, 1]]

    # ---------------------------------------------------------
    # 8. Package results
    # ---------------------------------------------------------
    params = {
        'placename': place_and_time['placename'],
        'xyt': xyt,
        'grid_resolution': grid_resolution,
        'cell_area': cell_area,
        'resampling_option': resampling_option,
        'year1': y1,
        'year2': y2
    }

    # ---------------------------------------------------------
    # 9. Optional summary output
    # ---------------------------------------------------------
    if verbose:
        print(place_and_time['placename'], ":", place_and_time['desc'])
        print("Coordinate range       :", xyt)
        print("Years                  :", y1, y2)
        print("Grid resolution will be:", grid_resolution)
        print("Rough dimensions (x,y) :", int(dxm / 1000), "by", int(abs(dym / 1000)), "km")
        print("Total cells (approx)   :", int(cellcount))
        print("Cell area              :", cell_area, "km²")
        print("Resampling strategy    :", resampling_option)

    return params

In [10]:
place_and_time = set_place_and_time('Lake_Manyara',year1='2020-01-01',year2='2025-12-31')

In [11]:
parameters = set_analysis_parameters(place_and_time,max_cells=10000,verbose = True)

{'x': (35.724, 35.929), 'y': (-3.814, -3.409), 'time': ('2020-01-01', '2025-12-31')}
Lake_Manyara : Lake_Manyara, Tanzania
Coordinate range       : {'x': (35.724, 35.929), 'y': (-3.814, -3.409), 'time': ('2020-01-01', '2025-12-31')}
Years                  : 2020 2025
Grid resolution will be: (280, 280)
Rough dimensions (x,y) : 20 by 40 km
Total cells (approx)   : 10566
Cell area              : 0.0784 km²
Resampling strategy    : nearest


In [12]:
products = {
    "tm": ["ls5_sr", "ls7_sr"],
    "oli": ["ls8_sr", "ls9_sr"],
    "msi": ["s2_l2a"],
}

In [18]:
# Sensor end dates (None = still active)
SENSOR_END_DATES = {
    "tm": "2023-12-31",  # Landsat 5/7
    "oli": None,         # Landsat 8/9 (ongoing)
    "msi": None          # Sentinel-2 (ongoing)
}

# Extract spatial + temporal bounds from parameters
x_range = tuple(parameters["xyt"]["x"])
y_range = tuple(parameters["xyt"]["y"])
start_time = parameters["xyt"]["time"][0]
end_time   = parameters["xyt"]["time"][1]

for sensor in products.keys():

    sensor_end = SENSOR_END_DATES[sensor]
    effective_end = min(end_time, sensor_end) if sensor_end else end_time

    if start_time > effective_end:
        print(f"Skipping {sensor} (no valid time range)")
        continue

    print(f"Loading {sensor}")
    print(f"time: {start_time} → {effective_end}")

    qa_band = measurements[sensor].get("qa") or measurements[sensor].get("pixel_quality")

    ds_list[sensor] = load_ard(
        dc=dc,
        products=products[sensor],
        x=x_range,
        y=y_range,
        time=(start_time, effective_end),
        output_crs="EPSG:6933",
        resolution=parameters["grid_resolution"],
        resampling=parameters["resampling_option"],
        verbose=True,
        dask_chunks={"x": 512, "y": 512},
        measurements=list(measurements[sensor].keys())
    ).rename(measurements[sensor])

Loading tm
time: 2020-01-01 → 2023-12-31
Using pixel quality parameters for USGS Collection 2
Finding datasets
    ls5_sr
    ls7_sr
Applying pixel quality/cloud mask
Re-scaling Landsat C2 data
Returning 276 time steps as a dask array
Loading oli
time: 2020-01-01 → 2025-12-31
Using pixel quality parameters for USGS Collection 2
Finding datasets
    ls8_sr


/opt/venv/lib/python3.12/site-packages/deafrica_tools/datahandling.py:565: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge([ds_data, ds_masks])


    ls9_sr
Applying pixel quality/cloud mask
Re-scaling Landsat C2 data
Returning 843 time steps as a dask array
Loading msi
time: 2020-01-01 → 2025-12-31
Using pixel quality parameters for Sentinel 2
Finding datasets
    s2_l2a


/opt/venv/lib/python3.12/site-packages/deafrica_tools/datahandling.py:565: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge([ds_data, ds_masks])


Applying pixel quality/cloud mask
Returning 1855 time steps as a dask array


/opt/venv/lib/python3.12/site-packages/deafrica_tools/datahandling.py:565: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge([ds_data, ds_masks])


### Apply TSM and CHLA algorithms

In [19]:
def calibrate(data):
    return(-2.0559+.5824*data)

In [20]:
def calibrate_chla(data):
    return(-27.342+2.0894*data)

In [21]:
suffix = ""  # no suffix
ALGORITHMS_CHLA, ALGORITHMS_TSM = set_wq_algorithms(suffix)

ds_dict = {}
ds_dict_chla = {}

for sensor in products.keys():
    print(sensor)
    print("Aligning water mask...")
    water_mask_aligned = ds.clear_water.reindex_like(ds_list[sensor], method="nearest")
    
    print("Applying Rayleigh correction...")
    ds_corrected = R_correction(ds_list[sensor], water_mask_aligned, instrument=sensor, drop=True)

    print("Applying TSM algorithm...")
    # Run only for OLI (if you only have OLI)
    tsm_ds = run_wq_algorithms(
        instrument_data={sensor: ds_corrected},  # instrument key must match algorithm dict
        algorithms_group=ALGORITHMS_TSM
    )

    print("Applying harmonisation...")
    tsm_ds = harmonize_wq_variables(tsm_ds)

    print("Applying normalisation...")
    tsm_ds = normalize_wq_variables(tsm_ds)

    print("Applying calibration...")
    for var in tsm_ds.data_vars:
        tsm_ds[var] = calibrate(tsm_ds[var])

    ds_dict[sensor] = tsm_ds

    print("Applying Chla algorithm...")
    # Run only for OLI (if you only have OLI)
    chla_ds = run_wq_algorithms(
        instrument_data={sensor: ds_corrected},  # instrument key must match algorithm dict
        algorithms_group=ALGORITHMS_CHLA
    )

    print("Applying harmonisation to Chla...")
    chla_ds = harmonize_wq_variables(chla_ds)

    print("Applying normalisation to Chla...")
    chla_ds = normalize_wq_variables(chla_ds)

    print("Applying calibration to Chla...")
    for var in chla_ds.data_vars:
        chla_ds[var] = calibrate_chla(chla_ds[var])

    ds_dict_chla[sensor] = chla_ds

    print(f"Finished {sensor}")

tm
Aligning water mask...
Applying Rayleigh correction...
Applying TSM algorithm...
Applying harmonisation...


/opt/venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: overflow encountered in power
  return self.func(*new_argspec)
/opt/venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: overflow encountered in power
  return self.func(*new_argspec)
/opt/venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: overflow encountered in power
  return self.func(*new_argspec)
/opt/venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: overflow encountered in power
  return self.func(*new_argspec)
/opt/venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: overflow encountered in power
  return self.func(*new_argspec)
/opt/venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: overflow encountered in power
  return self.func(*new_argspec)
/opt/venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: overflow encountered in power
  return self.func(*new_argspec)
/opt/v

Applying normalisation...
Applying calibration...
Applying Chla algorithm...
Applying harmonisation to Chla...
Applying normalisation to Chla...
Applying calibration to Chla...
Finished tm
oli
Aligning water mask...
Applying Rayleigh correction...


Variable oli01 expected but not found in the dataset - (non-fatal error) for instrument oli


Applying TSM algorithm...
Applying harmonisation...
Applying normalisation...
Applying calibration...
Applying Chla algorithm...
Applying harmonisation to Chla...
Applying normalisation to Chla...
Applying calibration to Chla...
Finished oli
msi
Aligning water mask...
Applying Rayleigh correction...


Variable msi01 expected but not found in the dataset - (non-fatal error) for instrument msi


Applying TSM algorithm...
Applying harmonisation...
Applying normalisation...
Applying calibration...
Applying Chla algorithm...
Applying harmonisation to Chla...
Applying normalisation to Chla...
Applying calibration to Chla...
Finished msi


In [2]:
tsm_ds = xr.concat(ds_dict.values(), dim="time").sortby("time")

NameError: name 'xr' is not defined

In [3]:
chla_ds = xr.concat(ds_dict_chla.values(), dim="time").sortby("time")

NameError: name 'xr' is not defined

In [4]:
tsm_da = tsm_ds.to_stacked_array(
new_dim="tsm_measures", # new dimension for stacking
sample_dims=["time", "y", "x"], # keep spatial+temporal dims
variable_dim="tsm_wq_vars", # names of the algorithms
name="tsm"
)

NameError: name 'tsm_ds' is not defined

In [ ]:
chla_da = chla_ds.to_stacked_array(
new_dim="chla_measures", # new dimension for stacking
sample_dims=["time", "y", "x"], # keep spatial+temporal dims
variable_dim="chla_wq_vars", # names of the algorithms
name="chla"
)

In [ ]:
tsm_final = tsm_da.median(dim="tsm_measures")
chla_final = chla_da.median(dim="chla_measures")

In [ ]:
tsi_final = compute_trophic_state_index(chla_final)

In [ ]:
tsm_monthly = tsm_final.resample(time="1M").mean()
chla_monthly = chla_final.resample(time="1M").mean()

In [ ]:
tsm_month_median = tsm_monthly.median(dim=("x", "y"))
chla_month_median = chla_monthly.median(dim=("x", "y"))

In [ ]:
# Calibrate yearly data
ds["tsm"] = calibrate(ds.tsm)
ds["chla"] = calibrate_chla(ds.chla)

In [ ]:
Q = np.array([0, 0.21, 0.79, 1])
quantile_vals = tsm_month_median.quantile(Q, skipna=True).values

# Labels and RGBA colors with alpha=0.1
labels = ["Low", "Normal", "High"]
colors = {
    "Low": "rgba(0, 0, 255, 0.1)",    # blue
    "Normal": "rgba(0, 128, 0, 0.1)", # green
    "High": "rgba(255, 0, 0, 0.1)"    # red
}

# Create classes array
classes = xr.full_like(tsm_month_median, np.nan, dtype=object)
valid = tsm_month_median.notnull()
class_index = np.digitize(tsm_month_median.values[valid], quantile_vals[1:-1])
classes.values[valid] = np.array(labels)[class_index]

# Build DataFrame with non-NaN values
df = pd.DataFrame({
    "Month": pd.to_datetime(tsm_month_median.time.values[valid]),  
    "TSM": tsm_month_median.values[valid],
    "Class": classes.values[valid]
})

# Format Time nicely
df["Month"] = df["Month"].dt.strftime("%Y-%m")

# Format TSM to 3 significant figures
df["TSM"] = df["TSM"].map(lambda x: f"{x:.3g}")

# Styling function with RGBA colors for alpha
def color_class(val):
    return f'background-color: {colors.get(val, "")}'

# Apply styling to Class column
df_styled = df.style.applymap(color_class, subset=["Class"])

df_styled

In [ ]:
# Step 2: Filter for 2020 onwards
time_mask = tsm_month_median.time.dt.year >= 2020
tsm_month_median_filtered = tsm_month_median.sel(time=time_mask)

# Step 3: Remove NaNs for monthly plot
valid = ~np.isnan(tsm_month_median_filtered.values)
x = tsm_month_median_filtered.time.values[valid]
y = tsm_month_median_filtered.values[valid]

labels = ["low", "normal", "high"]
colors = ["blue", "green", "red"]
alphas = [0.1, 0.1, 0.1]

plt.figure(figsize=(12,5))

# Step 4: Horizontal bands
plt.axhspan(quantile_vals[0], quantile_vals[1], color=colors[0], alpha=alphas[0], label="Low TSM")
plt.axhspan(quantile_vals[1], quantile_vals[2], color=colors[1], alpha=alphas[1], label="Normal TSM")
plt.axhspan(quantile_vals[2], quantile_vals[3], color=colors[2], alpha=alphas[2], label="High TSM")

# Step 5: Step plot for yearly spatial median, filtered for 2020 onwards
# yearly_mask = ds.tsm.time.dt.year >= 2017
# plt.step(
#     ds.tsm.time.values[yearly_mask],
#     ds.tsm.median(dim=("x","y")).values[yearly_mask],
#     where="post",
#     label="Yearly spatial median"
# )

# Step 6: Plot monthly median (skip NaNs)
plt.plot(x, y, marker="o", color="black", label="Monthly median TSM")

plt.xlabel("Time")
plt.ylabel("TSM")
plt.title("Monthly median TSM")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
Q = np.array([0, 0.21, 0.79, 1])
quantile_vals = chla_month_median.quantile(Q, skipna=True).values

# Labels and RGBA colors with alpha=0.1
labels = ["Low", "Normal", "High"]
colors = {
    "Low": "rgba(0, 0, 255, 0.1)",    # blue
    "Normal": "rgba(0, 128, 0, 0.1)", # green
    "High": "rgba(255, 0, 0, 0.1)"    # red
}

# Create classes array
classes = xr.full_like(chla_month_median, np.nan, dtype=object)
valid = chla_month_median.notnull()
class_index = np.digitize(chla_month_median.values[valid], quantile_vals[1:-1])
classes.values[valid] = np.array(labels)[class_index]

# Build DataFrame with non-NaN values
df = pd.DataFrame({
    "Month": pd.to_datetime(chla_month_median.time.values[valid]),  
    "chla": chla_month_median.values[valid],
    "Class": classes.values[valid]
})

# Format Time nicely
df["Month"] = df["Month"].dt.strftime("%Y-%m")

# Format chla to 3 significant figures
df["chla"] = df["chla"].map(lambda x: f"{x:.3g}")

# Styling function with RGBA colors for alpha
def color_class(val):
    return f'background-color: {colors.get(val, "")}'

# Apply styling to Class column
df_styled = df.style.applymap(color_class, subset=["Class"])

df_styled

In [ ]:
# Step 2: Filter for 2020 onwards
time_mask = chla_month_median.time.dt.year >= 2020
chla_month_median_filtered = chla_month_median.sel(time=time_mask)

# Step 3: Remove NaNs for monthly plot
valid = ~np.isnan(chla_month_median_filtered.values)
x = chla_month_median_filtered.time.values[valid]
y = chla_month_median_filtered.values[valid]

labels = ["low", "normal", "high"]
colors = ["blue", "green", "red"]
alphas = [0.1, 0.1, 0.1]

plt.figure(figsize=(12,5))

# Step 4: Horizontal bands
plt.axhspan(quantile_vals[0], quantile_vals[1], color=colors[0], alpha=alphas[0], label="Low TSM")
plt.axhspan(quantile_vals[1], quantile_vals[2], color=colors[1], alpha=alphas[1], label="Normal TSM")
plt.axhspan(quantile_vals[2], quantile_vals[3], color=colors[2], alpha=alphas[2], label="High TSM")

# Step 5: Step plot for yearly spatial median, filtered for 2020 onwards
yearly_mask = ds.tsm.time.dt.year >= 2017
plt.step(
    ds.chla.time.values[yearly_mask],
    ds.chla.median(dim=("x","y")).values[yearly_mask],
    where="post",
    label="Yearly spatial median"
)

# Step 6: Plot monthly median (skip NaNs)
plt.plot(x, y, marker="o", color="black", label="Monthly median CHLA")

plt.xlabel("Time")
plt.ylabel("CHLA")
plt.title("Monthly median CHLA")
plt.legend()
plt.grid(True)
plt.show()

***

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Africa data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Slack channel](http://slack.opendatacube.org/) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [Github](https://github.com/digitalearthafrica/deafrica-sandbox-notebooks).

**Compatible datacube version:** 

In [7]:
print(datacube.__version__)

1.8.4.dev52+g07bc51a5


**Last Tested:**

In [8]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

'2021-03-02'